# 02 — Analyze a clean claim package

Runs the `proClaimsV1` pro-mode analyzer over the `claim_auto_collision` sample (FNOL claim form + police report + repair estimate + damage photo), reasoning against the reference auto policy.

Pro mode does multi-input + multi-step reasoning, so the model can answer questions like _'does coverage apply?'_ and _'is the document set complete?'_ — not just verbatim extraction.

In [ ]:
from _lib import pro_service, SAMPLES_DIR, CU_OUTPUT_DIR
import json

manifest, files = pro_service.load_sample('claim_auto_collision')
print('Sample :', manifest.id, '-', manifest.title)
for p in files:
    print(f'  - {p.name:25s} ({p.stat().st_size:,} bytes)')

In [ ]:
result = pro_service.analyze_claims(files, sample_id='claim_auto_collision')
print(f'Analyzer  : {result.meta.analyzer_id}')
print(f'Elapsed   : {result.meta.elapsed_sec}s')
print()
for k, v in result.fields.model_dump().items():
    if v is not None:
        if isinstance(v, str) and len(v) > 120:
            v = v[:117] + '...'
        print(f'  {k:30s} {v}')

In [ ]:
# Persist the raw CU response for reference (used by 04_end_to_end and the README).
CU_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out = CU_OUTPUT_DIR / 'claim_auto_collision.json'
out.write_text(json.dumps(result.raw, indent=2, default=str), encoding='utf-8')
print('Saved:', out)

**Verdict.** The clean package should produce `CoverageApplies = Yes`, `PoliceReportPresent = Yes`, `DocumentSetCompleteness = Complete`, and a positive claims-handler verdict. If any of those are off, inspect the raw CU response in `reference/cu-output/claim_auto_collision.json`.